<a href="https://colab.research.google.com/github/AntonTian/AI-Project/blob/main/Model_Making.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Library Installation (NLP)**





In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install --upgrade torch torchvision transformers

**Library Installation (CV)**

In [ ]:
!pip install opencv-python-headless deepface

In [ ]:
!pip install lz4

**CV Module**

In [ ]:
import cv2
from deepface import DeepFace

image_path = "WhatsApp Image 2026-04-05 at 4.56.52 PM.jpeg"

print(f"\n[Face Analysis] : Processing image '{image_path}'...")
try:
    # Calling DeepFace with CNN Backend
    result = DeepFace.analyze(
        img_path=image_path,
        actions=['emotion'],
        enforce_detection=False,
        detector_backend='mtcnn'
    )

    emotions = result[0]['emotion']
    dominant = result[0]['dominant_emotion']

    print(f"Dominant emotion: {dominant.upper()}")

    # Arousal Formula
    arousal = (
        (emotions['surprise'] * 1.0) +
        (emotions['fear'] * 0.9) +
        (emotions['angry'] * 0.9) +
        (emotions['happy'] * 0.8) +
        (emotions['disgust'] * 0.4) +
        (emotions['neutral'] * 0.1) +
        (emotions['sad'] * 0.05)
    ) / 100.0

    print(f"Arousal result (Energy): {arousal:.2f}")

except Exception as e:
    print(f"Error: {e}")

**Spotify Dataset + EDA**

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
sns.set_style("white")
import matplotlib.pyplot as plt
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
file_path = 'dataset.csv'
df = pd.read_csv(file_path)

In [ ]:
pd.set_option('display.float_format', '{:.3f}'.format)
print(df.describe())

In [ ]:
df = df.drop(columns=['Unnamed: 0'])
df
df.info()

In [ ]:
df=df.dropna()

In [ ]:
df.isna().sum()

In [ ]:
df = df.drop_duplicates(subset=['track_name', 'artists'])
print(f"Total Unique Songs: {df.shape[0]}")

In [ ]:
df.shape

In [ ]:
plt.figure(figsize=(10, 8))
# Scatter plot Valence vs Energy
sns.scatterplot(data=df, x='valence', y='energy', alpha=0.1, color='blue')

# Making the quadrant line in the middle (0.5)
plt.axvline(0.5, color='red', linestyle='--')
plt.axhline(0.5, color='red', linestyle='--')

plt.title('Songs spread pattern based on Russel"s circumplex model (valence vs arousal)')
plt.xlabel('Valence (Negative -> Positive)')
plt.ylabel('Energy / Arousal (low -> High)')

# Quadrant Labels
plt.text(0.75, 0.75, 'Happy / Excited', fontsize=12, ha='center')
plt.text(0.25, 0.75, 'Angry / Tense', fontsize=12, ha='center')
plt.text(0.25, 0.25, 'Sad / Depressed', fontsize=12, ha='center')
plt.text(0.75, 0.25, 'Relaxed / Calm', fontsize=12, ha='center')

plt.show()

**NLP Module**

In [ ]:
import torch
print(torch.__version__)

**NLP Model (before + failed to detect sarcasm)**

In [ ]:
from transformers import pipeline

print("Loading NLP Module (RoBERTa)...")
# using a model that has trained with GoEmotions specifically
nlp_model = pipeline(
    "text-classification",
    model="SamLowe/roberta-base-go_emotions",
    top_k=None
)

# GoEmotions mapping -> VALENCE (0.0 -> 1.0)
valence_map = {
    # positive
    'joy': 1.0, 'love': 1.0, 'excitement': 0.9, 'gratitude': 0.9,
    'pride': 0.9, 'relief': 0.8, 'optimism': 0.8, 'admiration': 0.8,
    'caring': 0.7, 'approval': 0.7, 'desire': 0.7, 'amusement': 0.8,

    # neutral
    'surprise': 0.6, 'realization': 0.5, 'neutral': 0.5,
    'curiosity': 0.5, 'confusion': 0.4,

    # negative
    'annoyance': 0.3, 'disapproval': 0.3, 'embarrassment': 0.2,
    'nervousness': 0.2, 'fear': 0.1, 'sadness': 0.1, 'disappointment': 0.1,
    'remorse': 0.1, 'anger': 0.0, 'disgust': 0.0, 'grief': 0.0
}

def extract_valence_advanced(journal_text):
    print(f"\n[GoEmotions Analysis] : '{journal_text}'")

    # getting the probability from all 28 classes
    result = nlp_model(journal_text)[0]

    # getting the top 3 emotions for more context in the analysis
    result = result[:3]
    print(f"Detected Emotions: ")
    for emotion in result:
        print(f" - {emotion['label'].upper()} ({emotion['score']*100:.1f}%)")

    # weighted average calculation of valence
    # combining all of the probability of the 28 classes into their own valence
    total_valence = 0.0
    for prediction in result:
        emotion_name = prediction['label']
        probability = prediction['score']

        # Multiply the valence value with the generated probability
        total_valence += (valence_map[emotion_name] * probability)

    print(f"Valence Result: {total_valence:.3f}")
    return total_valence

if __name__ == '__main__':
    print("\n=== NLP GoEmotions Testing ===")

    # 1st test: mourning sentence
    extract_valence_advanced("I lost my dog today, I can't stop crying.")

    # 2nd test: proud & grateful
    extract_valence_advanced("I finally finished my Master's thesis! Thank you to my mentor.")

    # 3rd test: ambiguous
    extract_valence_advanced("Great, another rainy day stuck in my room with a broken laptop. Just what I needed to make my life perfect.")

**NLP Model (After)**

In [ ]:
from transformers import pipeline

print("Loading NLP Modules...")

# Custom Gatekeeper from Google Drive
print("-> Loading Custom Sarcasm Gatekeeper...")
sarcasm_model = pipeline("text-classification", model="/content/drive/MyDrive/my_sarcasm_model")

# Load the GoEmotions model
print("-> Loading GoEmotions Base Model...")
goemotions_model = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None)

# Valence mapping
valence_map = {
    'joy': 1.0, 'love': 1.0, 'excitement': 0.9, 'gratitude': 0.9, 'pride': 0.9,
    'relief': 0.8, 'optimism': 0.8, 'admiration': 0.8, 'caring': 0.7, 'approval': 0.7,
    'desire': 0.7, 'amusement': 0.8, 'surprise': 0.6, 'realization': 0.5, 'neutral': 0.5,
    'curiosity': 0.5, 'confusion': 0.4, 'annoyance': 0.3, 'disapproval': 0.3,
    'embarrassment': 0.2, 'nervousness': 0.2, 'fear': 0.1, 'sadness': 0.1,
    'disappointment': 0.1, 'remorse': 0.1, 'anger': 0.0, 'disgust': 0.0, 'grief': 0.0
}

def get_valence(text):
    print(f"\n" + "="*40)
    print(f"Analyzing Text: '{text}'")
    print("="*40)

    # --- Sarcasm Gatekeeper ---
    sarcasm_pred = sarcasm_model(text)[0]
    is_sarcastic = sarcasm_pred['label'] == 'LABEL_1'

    print(f"[Stage 1] Gatekeeper Scan: {'Sarcasm Detected!' if is_sarcastic else 'Clear (Literal)'}")

    # --- Emotion Extraction ---
    results = goemotions_model(text)[0]
    total_valence = sum(valence_map[pred['label']] * pred['score'] for pred in results)

    # This prevents the score from exceeding 1.0 if multiple positive emotions are detected
    total_valence = min(total_valence, 1.0)

    print(f"[Stage 2] GoEmotions Raw Valence: {total_valence:.3f}")

    # --- Mathematical Intervention ---
    final_valence = total_valence
    if is_sarcastic:
        # If the literal words trick GoEmotions into outputting a high valence (> 0.5)
        # Forcefully invert it into the negative spectrum (e.g., 0.8 becomes 0.2)
        if total_valence > 0.5:
            final_valence = 1.0 - total_valence
            print(f"[Stage 3] False Positive Detected: Inverting false positive ({total_valence:.3f}) -> ({final_valence:.3f})")
        else:
            print("[Stage 3] No intervention needed (Valence is already negative).")

    print(f"\n>>> Final Valence Score (X-Axis): {final_valence:.3f} <<<")
    return final_valence

In [ ]:
test_val = get_valence("I am so incredibly happy, I just got the best news ever!")
test_sarcasm = get_valence("Oh brilliant, another meeting that could have been an email. My absolute favorite.")

**Fusion Model + Recommendation System (Prototype)**

In [ ]:
from google.colab import files
import os
import pandas as pd
import numpy as np
import cv2

#Environment
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import warnings
warnings.filterwarnings("ignore")

# Import the libraries
from transformers import pipeline
from deepface import DeepFace

# NLP Module (Valence)
print("Loading NLP Module (GoEmotions)...")
nlp_model = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None)

valence_map = {
    'joy': 1.0, 'love': 1.0, 'excitement': 0.9, 'gratitude': 0.9, 'pride': 0.9,
    'relief': 0.8, 'optimism': 0.8, 'admiration': 0.8, 'caring': 0.7, 'approval': 0.7,
    'desire': 0.7, 'amusement': 0.8, 'surprise': 0.6, 'realization': 0.5, 'neutral': 0.5,
    'curiosity': 0.5, 'confusion': 0.4, 'annoyance': 0.3, 'disapproval': 0.3,
    'embarrassment': 0.2, 'nervousness': 0.2, 'fear': 0.1, 'sadness': 0.1,
    'disappointment': 0.1, 'remorse': 0.1, 'anger': 0.0, 'disgust': 0.0, 'grief': 0.0
}

def get_valence(text):
    print(f"\n[1] Analyzing Text: '{text}'")
    results = nlp_model(text)[0]
    total_valence = sum(valence_map[pred['label']] * pred['score'] for pred in results)
    print(f"    -> NLP Valence (Sumbu X): {total_valence:.3f}")
    return total_valence

# CV Module (Arousal)
def get_arousal(image_path):
    print(f"\n[2] Initializing face analysis: '{image_path}'")
    try:
        results = DeepFace.analyze(img_path=image_path, actions=['emotion'], enforce_detection=False, detector_backend='mtcnn')
        emotions = results[0]['emotion']

        # Arousal Formula
        arousal = (
            (emotions['surprise'] * 1.0) + (emotions['fear'] * 0.9) +
            (emotions['angry'] * 0.9) + (emotions['happy'] * 0.8) +
            (emotions['disgust'] * 0.4) + (emotions['neutral'] * 0.1) +
            (emotions['sad'] * 0.05)
        ) / 100.0

        print(f"    -> CV Arousal (Y-Axis): {arousal:.3f}")
        return arousal
    except Exception as e:
        print(f"    -> Failed to do face analysis: {e}")
        return 0.5 # Neutral value if it does fail

# Fusion & Recommendation Module (Euclidean Distance)
def get_recommendations(user_valence, user_arousal, df, top_k=5):
    print(f"\n[3] Finding matches via coordinates: [{user_valence:.3f}, {user_arousal:.3f}]")

    # Counts the Euclidean Distance from the user's input & with every song from the database
    # Spotify Feature: 'valence' (X) & 'energy' (Y)
    df['euclidean_distance'] = np.sqrt(
        (df['valence'] - user_valence)**2 +
        (df['energy'] - user_arousal)**2
    )

    df_unik = df.drop_duplicates(subset=['track_name', 'artists'], keep='first')

    # Sorting it from the shortest (the most similar) to the longest
    rekomendasi = df_unik.sort_values(by='euclidean_distance', ascending=True).head(top_k)
    return rekomendasi

# Execution of the main function
if __name__ == '__main__':
    print("\n" + "="*50)
    print("MoodiFY Online (Late Fusion)")
    print("="*50)

    # A. Load the Dataset
    print("\nInitializing Spotify DB...")
    df_songs = pd.read_csv('dataset.csv')

    # B. User Input
    print("\n" + "-"*50)
    print("Insert your data here")
    print("-"*50)

    # Manual text
    text_input = input("Describe your day today: ")

    # Picture upload
    print("\nUpload your picture here:")
    uploaded = files.upload()

    # Manual pic
    facial_input = list(uploaded.keys())[0]
    print(f"File '{facial_input}' has been received!")

    # C. Extraction Process
    val_x = get_valence(text_input)
    aro_y = get_arousal(facial_input)

    # D. Fusion & Recommendation
    recommendation_result = get_recommendations(val_x, aro_y, df_songs, top_k=5)

    # E. Results
    print("\n" + "="*50)
    print("Top 5 Music Recommendation")
    print("="*50)

    for idx, row in recommendation_result.iterrows():
        # Checking the column name from the dataset itself
        title = row.get('track_name', row.get('name', 'Unknown Track'))
        artist = row.get('artists', row.get('artist', 'Unknown Artist'))
        distance = row['euclidean_distance']

        print(f"{title} - {artist} (Distance: {distance:.3f})")

**Fusion Model + Recommendation System (Optimized)**

In [ ]:
from google.colab import drive, files
import os
import pandas as pd
import numpy as np
import cv2
import tensorflow as tf

# Force legacy Keras for DeepFace
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import warnings
warnings.filterwarnings("ignore")

from transformers import pipeline
from deepface import DeepFace
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input

print("\n" + "="*50)
print("Initializing MoodiFY")
print("="*50)

# Mount Drive for the NLP Gatekeeper
drive.mount('/content/drive')

# NLP Module
print("\n-> Loading Custom Sarcasm Gatekeeper...")
sarcasm_model = pipeline("text-classification", model="/content/drive/MyDrive/my_sarcasm_model")

print("-> Loading GoEmotions Base Model...")
nlp_model = pipeline("text-classification", model="SamLowe/roberta-base-go_emotions", top_k=None)

valence_map = {
    'joy': 1.0, 'love': 1.0, 'excitement': 0.9, 'gratitude': 0.9, 'pride': 0.9,
    'relief': 0.8, 'optimism': 0.8, 'admiration': 0.8, 'caring': 0.7, 'approval': 0.7,
    'desire': 0.7, 'amusement': 0.8, 'surprise': 0.6, 'realization': 0.5, 'neutral': 0.5,
    'curiosity': 0.5, 'confusion': 0.4, 'annoyance': 0.3, 'disapproval': 0.3,
    'embarrassment': 0.2, 'nervousness': 0.2, 'fear': 0.1, 'sadness': 0.1,
    'disappointment': 0.1, 'remorse': 0.1, 'anger': 0.0, 'disgust': 0.0, 'grief': 0.0
}

# Embedding the Neural Fusion Engine
print("\n-> Generating Psychological Rules & Training Fusion Engine...")
np.random.seed(42)
tf.random.set_seed(42)
inputs, targets = [], []

for _ in range(5000):
    text_v = np.random.uniform(0.0, 1.0)
    face_a = np.random.uniform(0.0, 1.0)

    if text_v < 0.4 and face_a > 0.6:     # Joker Effect
        target_v, target_e = text_v, text_v + 0.1
    elif text_v > 0.6 and face_a < 0.4:   # The Stoic
        target_v, target_e = text_v, text_v - 0.1
    else:                                 # Honest Expression
        target_v, target_e = text_v, face_a

    inputs.append([text_v, face_a])
    targets.append([np.clip(target_v, 0.0, 1.0), np.clip(target_e, 0.0, 1.0)])

# Build & Train on the fly instead of loading a file
fusion_model = Sequential([
    Input(shape=(2,)),
    Dense(16, activation='relu'),
    Dense(8, activation='relu'),
    Dense(2, activation='linear')
])
fusion_model.compile(optimizer='adam', loss='mse')
fusion_model.fit(np.array(inputs), np.array(targets), epochs=15, batch_size=64, verbose=0)
print("Fusion Engine Online!")

# Core Functions

def get_valence(text):
    print(f"\n[1] Analyzing Text: '{text}'")
    sarcasm_pred = sarcasm_model(text)[0]
    is_sarcastic = sarcasm_pred['label'] == 'LABEL_1'
    print(f"    -> Gatekeeper Scan: {'Sarcasm Detected' if is_sarcastic else 'Clear (Literal)'}")

    results = nlp_model(text)[0]
    total_valence = sum(valence_map[pred['label']] * pred['score'] for pred in results)
    total_valence = min(total_valence, 1.0)

    final_valence = total_valence
    if is_sarcastic and total_valence > 0.5:
        final_valence = 1.0 - total_valence
        print(f"    -> False Positive Detected: Inverting false positive ({total_valence:.3f}) -> ({final_valence:.3f})")

    print(f"    -> Final NLP Valence (X-Axis): {final_valence:.3f}")
    return final_valence


def get_arousal(image_path):
    print(f"\n[2] Initializing face analysis...")
    try:
        results = DeepFace.analyze(img_path=image_path, actions=['emotion'], enforce_detection=False, detector_backend='mtcnn')
        emotions = results[0]['emotion']

        arousal = (
            (emotions['surprise'] * 1.0) + (emotions['fear'] * 0.9) +
            (emotions['angry'] * 0.9) + (emotions['happy'] * 0.8) +
            (emotions['disgust'] * 0.4) + (emotions['neutral'] * 0.1) +
            (emotions['sad'] * 0.05)
        ) / 100.0

        print(f"    -> CV Arousal (Y-Axis): {arousal:.3f}")
        return arousal
    except Exception as e:
        print(f"    -> Failed to do face analysis: {e}")
        return 0.5


def get_recommendations(user_valence, user_arousal, df, top_k=5):
    print(f"\n[3] Engaging Neural Fusion Engine...")

    input_data = np.array([[user_valence, user_arousal]])
    predicted_coords = fusion_model.predict(input_data, verbose=0)[0]

    target_v = np.clip(predicted_coords[0], 0.0, 1.0)
    target_e = np.clip(predicted_coords[1], 0.0, 1.0)

    print(f"    -> Adjusting Target Coordinates: [Valence: {target_v:.3f}, Energy: {target_e:.3f}]")

    df['euclidean_distance'] = np.sqrt(
        (df['valence'] - target_v)**2 + (df['energy'] - target_e)**2
    )

    df_unik = df.drop_duplicates(subset=['track_name', 'artists'], keep='first')
    rekomendasi = df_unik.sort_values(by='euclidean_distance', ascending=True).head(top_k)
    return rekomendasi


# Main Function
if __name__ == '__main__':
    try:
        df_songs = pd.read_csv('dataset.csv')
    except FileNotFoundError:
        print("\nError: 'dataset.csv' not found! Please upload your Spotify dataset to Colab.")
        exit()

    print("\n" + "-"*50)
    print("USER INPUT")
    print("-" * 50)

    text_input = input("Describe your day today: ")

    print("\nUpload your picture here:")
    uploaded = files.upload()
    facial_input = list(uploaded.keys())[0]

    val_x = get_valence(text_input)
    aro_y = get_arousal(facial_input)

    recommendation_result = get_recommendations(val_x, aro_y, df_songs, top_k=5)

    print("\n" + "="*50)
    print("Top 5 Music Recommendations")
    print("="*50)

    for idx, row in recommendation_result.iterrows():
        title = row.get('track_name', row.get('name', 'Unknown Track'))
        artist = row.get('artists', row.get('artist', 'Unknown Artist'))
        distance = row['euclidean_distance']
        print(f"{title} - {artist} (Match Score: {distance:.3f})")